# LlamaIndex
LlamaIndex 一个专门针对数据（Data-centric）的索引与检索增强框架

## LlamaIndex 索引架构与复合 Query Engine
1. 掌握 LlamaIndex 数据抽象与三大索引：理解 `Document` 到 `Node` 的映射逻辑，掌握 `VectorStoreIndex`（语义索引）、`SummaryIndex`（顺序/摘要索引） 与 `KnowledgeGraphIndex`（知识图谱索引） 的适用场景。
2. 攻克复合查询（`RouterQueryEngine` & `SubQuestionQueryEngine`）：学习如何让框架根据用户 Query 动态选择最合适的索引，或自动将跨文档复合问题拆解为多个子查询并行执行。
3. 对比 LangChain 与 LlamaIndex 技术选型：明确在真实企业项目中何时选择 LangChain，何时选择 LlamaIndex，或两者如何组合使用。

#### 数据节点（Nodes）与复合查询引擎
1. 从 Document 到 Node 的数据流

    在 LlamaIndex 中，数据流的处理非常严谨且标准化：
   * Reader / Loading：加载原始数据（PDF、API、数据库）生成 `Document` 对象。
   * Node Parser (Transformations)：将 `Document` 切分并转化为 Node（数据节点）。`Node` 不仅包含文本，还包含丰富的关系元数据（如 `prev_node`、`next_node`、`parent_node`）。
   * Index 构建：在 `Node` 集合之上构建特定的数据结构索引。

2. LlamaIndex 三大核心索引对比

    | 索引类型                    | 数据组织方式                             | 核心优势             | 最佳适用场景                           |
    |-------------------------|------------------------------------|------------------|----------------------------------|
    | **VectorStoreIndex**    | 将 Node 向量化存入向量库                    | 高精确度的语义相似度匹配     | 查找具体的事实、条款、细节定义（“ERR_9021 如何解决”） |
    | **SummaryIndex**        | 将所有 Node 串联成顺序链表                   | 遍历所有 Node 提取全貌信息 | 全局总结、归纳（“这份 100 页报告总结了哪三大风险”）    |
    | **KnowledgeGraphIndex** | 自动提取实体 (Entity) 与关系 (Triplet) 构建图谱 | 跨文档的隐式关联分析与推理    | 复杂的逻辑推演（“A 公司与 B 公司的股权与高管重叠关系”）  |

3. 高级复合查询引擎：SubQuestionQueryEngine

    在面对复杂的对比型或多跨度问题时（例如：“对比分析产品 A 的定价策略与产品 B 的交付周期”）：
   * 单一向量检索只能命中部分文本，导致回答不完整。
   * SubQuestionQueryEngine 的工作机制：
     1. 调用 LLM 将主问题拆解为若干子问题：
        * 子问题 1：“产品 A 的定价策略是什么？” -> 路由至 `ProductA_Index`
        * 子问题 2：“产品 B 的交付周期是多久？” -> 路由至 `ProductB_Index`
     2. 并行执行子查询并拉取对应的 Context。
     3. 将所有子问题的解答汇聚，生成最终的高质量综合回答。



下面的代码展示了如何使用 LlamaIndex（轻量级 Python 内存示例）搭建包含 RouterQueryEngine 和 SubQuestionQueryEngine 的复合查询框架：

In [ ]:
import os
from llama_index.core import VectorStoreIndex, SummaryIndex, Document
from llama_index.core.node_parser import SentenceSplitter
from llama_index.core.tools import QueryEngineTool, ToolMetadata
from llama_index.core.query_engine import RouterQueryEngine, SubQuestionQueryEngine
from llama_index.core.selectors import LLMSingleSelector

# --- 1. 准备测试文档数据 ---
doc_a = Document(
    text="【产品 A 规格说明】\n售价为 999 美元/年。功能特色：支持高并发 Agent 部署，接口响应延迟小于 50ms。适用领域：金融与高频交易。",
    metadata={"doc_name": "Product_A"}
)

doc_b = Document(
    text="【产品 B 规格说明】\n售价为 299 美元/年。功能特色：极简开箱即用，内置可视化 Workflow 拖拽编辑器。交付周期：下单后 24 小时内开通账号。",
    metadata={"doc_name": "Product_B"}
)

# --- 2. 构建数据节点与不同索引 ---
node_parser = SentenceSplitter(chunk_size=200, chunk_overlap=20)
nodes_a = node_parser.get_nodes_from_documents([doc_a])
nodes_b = node_parser.get_nodes_from_documents([doc_b])

# 分别对产品 A 和产品 B 构建 VectorStoreIndex
index_a = VectorStoreIndex(nodes_a)
index_b = VectorStoreIndex(nodes_b)

# 构建一个针对全量文档的 SummaryIndex (用于回答全局总结类问题)
summary_index = SummaryIndex(nodes_a + nodes_b)

# --- 3. 封装 QueryEngine 工具节点 ---
engine_a = index_a.as_query_engine()
engine_b = index_b.as_query_engine()
summary_engine = summary_index.as_query_engine()

query_engine_tools = [
    QueryEngineTool(
        query_engine=engine_a,
        metadata=ToolMetadata(
            name="product_a_tool",
            description="用于查询产品 A 的价格、性能指标、延迟以及适用领域等细节。"
        ),
    ),
    QueryEngineTool(
        query_engine=engine_b,
        metadata=ToolMetadata(
            name="product_b_tool",
            description="用于查询产品 B 的价格、交付周期、易用性以及可视化编辑器等细节。"
        ),
    ),
]

# --- 4. 方案 A：RouterQueryEngine (根据问题自动二选一) ---
router_engine = RouterQueryEngine(
    selector=LLMSingleSelector.from_defaults(),
    query_engine_tools=query_engine_tools
)

# --- 5. 方案 B：SubQuestionQueryEngine (拆解复杂跨文档问题) ---
# 注意：在真实环境下需传入大模型实例如 ChatOpenAI / Ollama
# sub_question_engine = SubQuestionQueryEngine.from_defaults(
#     query_engine_tools=query_engine_tools
# )

if __name__ == "__main__":
    print("🚀 运行 RouterQueryEngine 单路由测试:")
    query_1 = "产品 A 的接口延迟表现怎么样？"
    print(f"❓ 查询: {query_1}")
    # 框架将自动根据 Tool Description 路由到 product_a_tool
    response_1 = router_engine.query(query_1)
    print(f"💡 回答: {response_1}\n")

    print("🚀 运行跨文档拆解测试逻辑 (Sub-Question 理论展示):")
    query_2 = "对比产品 A 和产品 B 的价格与交付时间"
    print(f"❓ 跨文档多子问题查询: {query_2}")
    print("👉 SubQuestionQueryEngine 会自动生成两条子查询:")
    print("   ↳ 子查询 1: '产品 A 的价格是多少？' (路由至 product_a_tool)")
    print("   ↳ 子查询 2: '产品 B 的价格和交付时间是多少？' (路由至 product_b_tool)")
    print("👉 最终拼接上下文，生成对比报告。")

1. LangChain 与 LlamaIndex 技术选型指南：
    * 在团队开发复杂 AI 系统时，经常面临框架选型。
    * 工程思考：如果你的项目核心需求是“打通内部复杂的多源非结构化文档（PDF、 Notion、数据库），搭建高性能 RAG 知识库”，而另一个项目的核心需求是“打造一个能自动调用 API、循环思考并执行复杂多步骤任务的 Agent 工作流”，你会分别为这两个项目选择哪个框架？为什么？
        * 高性能 RAG 知识库 → 选 LlamaIndex
            * LlamaIndex 是"数据检索专家"，专为 RAG 而生。
        * 复杂多步骤 Agent 工作流 → 选 LangChain + LangGraph
            * 核心理由：LangChain 是"执行层引擎"，LangGraph 是"状态机编排器"，两者组合是复杂 Agent 的生产级标准方案。

2. Node 节点关系链的应用：
    * LlamaIndex 的 `Node` 结构天然保存了 `prev_node` 和 `next_node` 指针。
    * 思考：当某个 Chunk 命中了关键信息，但我们需要补充“该 Chunk 紧邻的上文和下文”时，LlamaIndex 如何利用节点关系链（`Next/Previous Node Relationship`）免去重新进行向量搜索的开销？
